# Reproducing Chandravadia et al. (2020) from dandiset 000004

**Paper.** Chandravadia N, Liang D, Schjetnan AGP, et al. (2020). A NWB-based dataset and processing
pipeline of human single-neuron activity during a declarative memory task. *Scientific Data* 7, 78.
[doi:10.1038/s41597-020-0415-9](https://doi.org/10.1038/s41597-020-0415-9). The claims file
(`claims/chandravadia-2020.yaml`) was written from the full text of the open access version of record
(Methods, Data Records, Technical Validation, figure legends, Tables 1 and 2 and Online-only Table 1).

**Dataset.** DANDI:000004, version **0.220126.1852**
([doi:10.48324/dandi.000004/0.220126.1852](https://doi.org/10.48324/dandi.000004/0.220126.1852)).
This version was asked for in the instructions for this attempt. The paper cites the data on the Open
Science Framework (doi:10.17605/OSF.IO/HV7JA), not a DANDI version; 0.220126.1852 is the most recent
published version, created a few seconds after 0.220126.1851 on 2022-01-26; the notebook shows that the two
versions list the same assets. Every file is streamed from the archive; the stimulus images, which make up most of each file,
are never read.

**Other inputs.** Online-only Table 1 of this paper and Table 1 of Faraut et al. (2018,
doi:10.1038/sdata.2018.10), which lists the sessions of the earlier release, are read from the Europe PMC
full text of the two open access articles (PMC7055261 and PMC5810422). The method details follow the
claims file, and where it is silent, the paper and its code (github.com/rutishauserlab/recogmem-release-NWB
at commit 1d95162); `chandravadia.py` says where each comes from.

| Claim | Tested here |
|---|---|
| counts-neurons-subjects | tested |
| counts-sessions | tested |
| table1-demographics | tested |
| task-structure | tested |
| new-old-label-coding | tested |
| ttl-codes | tested |
| experiment-ids | tested |
| session-dates | tested |
| units-table | tested |
| electrodes-table | tested |
| isi-criterion | tested |
| fig5a-mtl-locations | tested |
| fig5c-auc-all | tested |
| fig5c-auc-sites | tested |
| fig5b-roc-asymmetric | tested |
| fig5d-confidence-accuracy | tested |
| vs-cells-proportion | tested |
| ms-cells-proportion | tested |
| dual-cells-proportion | tested |
| fig5f-site-proportions | tested |
| twh-sessions | tested |
| previous-release-link | tested (uses Table 1 of Faraut et al. 2018) |
| previous-release-additions | tested (uses Table 1 of Faraut et al. 2018) |
| fig4-example-vs-neuron | tested |
| fig3-example-vs-neuron | blocked: session 144 is not in the dandiset |

In [ ]:
import time
import resource
import hashlib
import warnings
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

import chandravadia as C

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
started = time.time()

## The files and what they hold

In [ ]:
meta = C.version_metadata()
assets = C.list_assets()
print(f"dandiset {C.DANDISET} version {C.VERSION}, doi {meta['doi']}")
print("assets:", len(assets), " total size %.2f GB" % (assets['size'].sum() / 1e9))
print("archive's summary: subjects", meta["assetsSummary"]["numberOfSubjects"], " files", meta["assetsSummary"]["numberOfFiles"])
other = C.list_assets("0.220126.1851")
same = set(zip(other.path, other["size"], other.asset_id)) == set(zip(assets.path, assets["size"], assets.asset_id))
print("version 0.220126.1851, published the same day:", len(other), "assets; same paths, sizes and asset ids as", C.VERSION, ":", same)
t0 = time.time()
sessions = C.read_all(assets.asset_id)
print(f"read {len(sessions)} files in {time.time() - t0:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")
for s, p in zip(sessions, assets.path):
    s["path"] = p
    lab, sid = C.parse_identifier(s["identifier"])
    s["label"], s["sid"] = C.ident_label(lab), sid
    s["site"] = "TWH" if s["label"].startswith("T") else "CS"
s0 = sessions[0]
print("\nOne file,", s0["path"])
print("  trials columns:", s0["trial_columns"])
print("  electrodes columns:", s0["electrode_columns"])
print("  units columns:", s0["unit_columns"])
print("  subject fields:", s0["subject"])
print("  stimulus series:", s0["stimulus_series"], " devices:", s0["devices"])
print("  data_collection:", repr(s0["data_collection"]))
print("  new_old_labels_recog description:", repr(s0["trial_descriptions"]["new_old_labels_recog"]))
print("  every file has the same trials, electrodes and units columns:",
      len({tuple(s["trial_columns"]) for s in sessions}) == 1, len({tuple(s["electrode_columns"]) for s in sessions}) == 1,
      len({tuple(s["unit_columns"]) for s in sessions}) == 1)

## Data check

Counts by site, subject and session, the trial and unit counts, and the ranges and units of every
variable the analysis uses. The checks of the description against the values follow the table.

In [ ]:
rows = []
for s in sessions:
    tr = s["trials"]
    r = C.recog(tr)
    rows.append(dict(file=s["path"].split("/")[1][:48], identifier=s["identifier"], subject_id=s["subject"]["subject_id"],
                     label=s["label"], session=s["sid"], site=s["site"], start=s["session_start_time"][:10],
                     n_trials=len(tr), n_learn=int((tr.stim_phase == "learn").sum()), n_recog=len(r),
                     n_units=len(s["units"]), n_electrodes=len(s["electrodes"]),
                     variant=",".join(map(str, sorted(C.variant_of(tr.external_image_file)))),
                     exp_ids=",".join(str(int(x)) for x in np.unique(s["expids"]))))
sess = pd.DataFrame(rows)
print(sess.drop(columns="file").to_string())
print("\nfiles", len(sess), " distinct subject_id", sess.subject_id.nunique(), " distinct (subject, session)",
      len(set(zip(sess.label, sess.session))))
print("by site:", sess.groupby("site").agg(files=("session", "size"), subjects=("label", "nunique"), units=("n_units", "sum")).to_dict("index"))
print("subject_id agrees with the identifier's label in every file:",
      all(C.table_label(a) == b for a, b in zip(sess.subject_id, sess.label)))
print("trials per file:", sess.n_trials.value_counts().to_dict(), " learn:", sess.n_learn.value_counts().to_dict(),
      " recog:", sess.n_recog.value_counts().to_dict())
print("units per file: min", sess.n_units.min(), " median", sess.n_units.median(), " max", sess.n_units.max(), " total", sess.n_units.sum())

allr = pd.concat([C.recog(s["trials"]).assign(identifier=s["identifier"]) for s in sessions])
alll = pd.concat([s["trials"][s["trials"].stim_phase == "learn"] for s in sessions])
print("\nrecognition response_value:", sorted(allr.response_value.unique()), " learning response_value:", sorted(alll.response_value.unique()))
print("new_old_labels_recog values, recognition:", allr.new_old_labels_recog.value_counts().to_dict(),
      " learning:", alll.new_old_labels_recog.value_counts().to_dict())
print("stimCategory values:", sorted(allr.stimCategory.unique()), " category names:", sorted(pd.concat([s["trials"].category_name for s in sessions]).unique()))
print("stim_phase values:", sorted(pd.concat([s["trials"].stim_phase for s in sessions]).unique()))
dur = pd.concat([s["trials"].stop_time - s["trials"].start_time for s in sessions])
print("stimulus duration stop_time - start_time (s): %.3f to %.3f, median %.3f" % (dur.min(), dur.max(), dur.median()))
print("start_time equals stim_on_time in every trial:", all(np.allclose(s["trials"].start_time, s["trials"].stim_on_time) for s in sessions))

spk_rng = []
for s in sessions:
    allsp = np.concatenate([x for x in s["spikes"] if len(x)])
    spk_rng.append((allsp.min() - s["trials"].start_time.min(), allsp.max() - s["trials"].delay2_time.max()))
spk_rng = np.array(spk_rng)
print("\nspike times (s, the file's clock): first spike minus first trial onset %.1f to %.1f s; last spike minus last trial end %.1f to %.1f s"
      % (spk_rng[:, 0].min(), spk_rng[:, 0].max(), spk_rng[:, 1].min(), spk_rng[:, 1].max()))
U = pd.concat([s["units"].assign(identifier=s["identifier"]) for s in sessions])
rates = np.array([len(sp) / (s["trials"].delay2_time.max() - s["trials"].start_time.min()) for s in sessions for sp in s["spikes"]])
print("mean firing rate over the task (Hz): median %.2f, range %.3f to %.1f" % (np.median(rates), rates.min(), rates.max()))
print("waveform sampling rate (Hz):", U.waveform_sampling_rate.value_counts().to_dict(), " samples per waveform:",
      U.waveform_mean_encoding_samples.value_counts().to_dict())
E = pd.concat([s["electrodes"].assign(identifier=s["identifier"]) for s in sessions])
print("electrode locations:", E.location.value_counts().to_dict())
print("electrode filtering:", E.filtering.value_counts().to_dict())
print("MNI x range %.1f to %.1f mm, y %.1f to %.1f, z %.1f to %.1f; impedance non-missing: %d of %d"
      % (E.x.min(), E.x.max(), E.y.min(), E.y.max(), E.z.min(), E.z.max(), E.imp.notna().sum(), len(E)))

Checks of the description against the data, from the output above and the cells below:

- The archive lists 87 files and 59 subjects; the files hold 87 distinct (subject, session) pairs and 59
  subject_ids, and each file's subject_id names the same subject as its identifier.
- The paper says each session has 100 encoding and 100 recognition trials; the counts above show
  sessions with 50 or 97 learning trials (claim task-structure).
- The response codes, the label coding, the TTL codes and the experiment ids are checked in their claims.
- Spike times are in seconds on the clock of the trial table: they start within seconds of the first trial
  and end with the task. The next cell checks the alignment with the trials.

### Do the spikes share the clock of the trial table?

Two checks. First, the TTL stimulus onsets (code 1) of the events stream are compared with the trials'
start times. Second, a contrast that an offset would destroy: the fraction of units whose recognition
firing rate differs across the 5 categories (one-way ANOVA, p < 0.05) in 1 s windows starting at
different times relative to stimulus onset. If the spikes and the trials share a clock, the fraction is
near the 5% false positive rate before onset and rises after it.

In [ ]:
mism = []
for s in sessions:
    on_ev = s["events_t"][s["events"] == 1]
    st = s["trials"].start_time.values
    n = min(len(on_ev), len(st))
    mism.append(dict(identifier=s["identifier"], ttl1=len(on_ev), trials=len(st),
                     max_abs_diff_ms=1000 * np.max(np.abs(on_ev[:n] - st[:n])) if n else np.nan))
mism = pd.DataFrame(mism)
print("files whose TTL-1 count equals the trial count:", (mism.ttl1 == mism.trials).sum(), "of", len(mism))
print("largest |TTL-1 time - start_time| over files where counts agree: %.3f ms" % mism[mism.ttl1 == mism.trials].max_abs_diff_ms.max())
print(mism[mism.ttl1 != mism.trials].to_string())

offsets = np.arange(-2.0, 2.01, 0.25)
frac = []
per_session = []
for s in sessions:
    r = C.recog(s["trials"])
    on, cat = r.start_time.values, r.stimCategory.values
    ps = np.array([[C.anova_p(C.window_rates(sp, on, (o, o + 1.0)), cat) for o in offsets] for sp in s["spikes"]])
    frac.append(ps)
    pre = np.nansum(ps[:, offsets == -1.25] < 0.05)
    post = np.nansum(ps[:, np.isclose(offsets, 0.25)] < 0.05)
    per_session.append(dict(identifier=s["identifier"], units=len(ps), sel_pre=int(pre), sel_post=int(post)))
frac = np.vstack(frac)
print("\nfraction of units category selective, by window start relative to onset (window 1 s):")
print(pd.Series(np.nanmean(frac < 0.05, axis=0).round(3), index=offsets).to_string())
ps_df = pd.DataFrame(per_session)
print("\nper session, units selective in the window -1.25 to -0.25 s against 0.25 to 1.25 s:")
print("  sessions with more selective units after onset:", (ps_df.sel_post > ps_df.sel_pre).sum(),
      " equal:", (ps_df.sel_post == ps_df.sel_pre).sum(), " fewer:", (ps_df.sel_post < ps_df.sel_pre).sum(), " of", len(ps_df))
print("  sessions where the contrast cannot show alignment (no selective unit after onset):",
      ps_df[ps_df.sel_post == 0].identifier.tolist())
fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(offsets, np.nanmean(frac < 0.05, axis=0), "o-")
ax.axhline(0.05, color="k", lw=0.5)
ax.set_xlabel("window start relative to stimulus onset (s)")
ax.set_ylabel("fraction category selective")
plt.show()

### Recordings deposited more than once

Every unit's spike train is hashed, and trains found in more than one file are listed by pair of files.
The trial tables of each pair are compared by phase, and the time span of the shared spike trains is given.

In [ ]:
owners = {}
for s in sessions:
    for i, sp in enumerate(s["spikes"]):
        owners.setdefault(hashlib.md5(np.asarray(sp).tobytes()).hexdigest(), []).append((s["identifier"], i))
dups = [v for v in owners.values() if len(v) > 1]
pairs = Counter(tuple(sorted({x[0] for x in v})) for v in dups)
byid = {s["identifier"]: s for s in sessions}
for (a, b), n in pairs.items():
    sa, sb = byid[a], byid[b]
    print(f"{a} ({len(sa['units'])} units) and {b} ({len(sb['units'])} units): {n} identical spike trains;"
          f" variants {sorted(C.variant_of(sa['trials'].external_image_file))} and {sorted(C.variant_of(sb['trials'].external_image_file))}")
    for ph in ("learn", "recog"):
        ta, tb = sa["trials"][sa["trials"].stim_phase == ph], sb["trials"][sb["trials"].stim_phase == ph]
        same = len(ta) == len(tb) and np.allclose(ta.start_time, tb.start_time) and np.array_equal(ta.external_image_file, tb.external_image_file)
        print(f"  {ph}: {a} {ta.start_time.min():.0f} to {ta.start_time.max():.0f} s, {b} {tb.start_time.min():.0f} to {tb.start_time.max():.0f} s;"
              f" identical trials (onsets and images): {same}")
    shared = np.concatenate([sp for sp in sa["spikes"] if hashlib.md5(np.asarray(sp).tobytes()).hexdigest() in
                             {hashlib.md5(np.asarray(x).tobytes()).hexdigest() for x in sb["spikes"]}])
    print(f"  the shared spike trains span {shared.min():.0f} to {shared.max():.0f} s")
dup_keys = {k for k, v in owners.items() if len(v) > 1}
n_dup_rows = sum(len(v) - 1 for v in dups)
print("unit rows:", sum(len(s["spikes"]) for s in sessions), " distinct spike trains:", len(owners), " rows that repeat another file's train:", n_dup_rows)
# the unit rows kept when duplicated trains are counted once: the first occurrence
seen, keep = set(), {}
for s in sessions:
    ks = []
    for sp in s["spikes"]:
        h = hashlib.md5(np.asarray(sp).tobytes()).hexdigest()
        ks.append(h not in seen)
        seen.add(h)
    keep[s["identifier"]] = np.array(ks)

The three pairs are consecutive sessions of one subject on one clock: the shared spike trains span both
sessions' trials. In H42_54 and H42_55 the encoding block is the same and the recognition blocks differ; in
the other two pairs both blocks differ. So these are the same neurons recorded during two sessions, each
deposited with both. They are counted per file, as the paper counts neurons per session, and counted once
as a variant.

## Reference tables from the papers

Online-only Table 1 of this paper (subjects, sessions, variants, age, sex, diagnosis) and Table 1 of
Faraut et al. (2018), the earlier release, read from the Europe PMC full text.

In [ ]:
paper_tables = C.europepmc_tables(C.PAPER_PMC)
T1 = C.session_table(paper_tables["Online-only Table 1"])
print("Online-only Table 1:", len(T1), "sessions,", T1.label.nunique(), "subjects; totals row:",
      [c for c in paper_tables["Online-only Table 1"][-1]])
faraut_tables = C.europepmc_tables(C.FARAUT_PMC)
F1 = C.session_table(faraut_tables["Table 1"])
print("Faraut et al. 2018 Table 1:", len(F1), "sessions,", F1.label.nunique(), "subjects")
print(T1.head(3).to_string())

## counts-neurons-subjects

> The dataset holds 1863 single neurons recorded from the medial temporal lobes of 59 subjects.

Criterion: neurons are the unit rows over all files, subjects the distinct subject_id values; reading (a)
every row, (b) spike trains deposited in two files counted once. Both within 10% under one reading.

In [ ]:
n_rows = int(sess.n_units.sum())
n_once = int(sum(k.sum() for k in keep.values()))
n_subj = sess.subject_id.nunique()
print(f"(a) unit rows: {n_rows} (expected 1863 +/- 186.3; difference {n_rows - 1863:+d})")
print(f"(b) duplicated trains counted once: {n_once} (difference {n_once - 1863:+d})")
print(f"subjects (distinct subject_id): {n_subj} (expected 59 +/- 5.9)")
print("units by subject: min", sess.groupby("label").n_units.sum().min(), " max", sess.groupby("label").n_units.sum().max())

**Result: reproduced.** Under reading (a) the files hold 1864 unit rows, one more than the paper's 1863,
and 59 subjects. Counting the 66 duplicated spike trains once gives 1798, also within tolerance.

## counts-sessions

> The patients performed 87 sessions, listed by subject and session id in Online-only Table 1 (the
> Methods give 89).

Criterion: the set of (subject, session id) pairs from the file identifiers against Online-only Table 1.

In [ ]:
files_set = set(zip(sess.label, sess.session))
table_set = set(zip(T1.label, T1.session))
print("pairs in the files:", len(files_set), " in Online-only Table 1:", len(table_set))
print("in the files only:", sorted(files_set - table_set), " in the table only:", sorted(table_set - files_set))
print("sets equal:", files_set == table_set, "; the Methods' 89 differs from the count by", 89 - len(files_set))

**Result: reproduced.** The 87 (subject, session) pairs of the files are exactly the 87 of Online-only
Table 1. The Methods' 89 is two more than either.

## table1-demographics

> Online-only Table 1 gives the age and sex of every subject; there are 25 female and 34 male subjects.

Criterion: age and sex compared subject by subject; reproduced when every subject matches.

In [ ]:
subj_files = pd.DataFrame([dict(label=s["label"], age=s["subject"]["age"], sex=s["subject"]["sex"],
                                diagnosis_file=s["subject"]["description"]) for s in sessions])
per_subj_files = subj_files.groupby("label").agg(age_file=("age", lambda x: ",".join(sorted(set(x)))),
                                                 sex_file=("sex", lambda x: ",".join(sorted(set(x)))),
                                                 diagnosis_file=("diagnosis_file", lambda x: ",".join(sorted(set(x)))))
per_subj_table = T1.groupby("label").agg(age_table=("age", "first"), sex_table=("sex", "first"), diagnosis_table=("diagnosis", "first"))
cmp = per_subj_files.join(per_subj_table, how="outer")
cmp["age_match"] = cmp.age_file == cmp.age_table.astype(str)
cmp["sex_match"] = cmp.sex_file == cmp.sex_table
print("files with more than one age or sex for a subject:", int(cmp.age_file.str.contains(",").sum() + cmp.sex_file.str.contains(",").sum()))
print("subjects:", len(cmp), " age matches:", int(cmp.age_match.sum()), " sex matches:", int(cmp.sex_match.sum()),
      " both:", int((cmp.age_match & cmp.sex_match).sum()))
print(cmp[~(cmp.age_match & cmp.sex_match)][["age_file", "age_table", "sex_file", "sex_table"]].to_string())
print("\nfemale/male in the files:", (cmp.sex_file == "F").sum(), (cmp.sex_file == "M").sum(),
      "; in the table:", (cmp.sex_table == "F").sum(), (cmp.sex_table == "M").sum(), "(expected 25 and 34)")
print("\ndiagnosis, side by side (first 12 subjects and every one that differs in words):")
diff_diag = cmp[cmp.diagnosis_file.str.lower() != cmp.diagnosis_table.str.lower()]
print(cmp[["diagnosis_file", "diagnosis_table"]].head(12).to_string())
print("subjects whose diagnosis text differs:", len(diff_diag), "of", len(cmp))
print(diff_diag[["diagnosis_file", "diagnosis_table"]].to_string())

**Result: partially reproduced.** 56 of 59 subjects match the table in age and sex. H33 is 30 in the files
and 29 in the table, C24 is 57 against 47, and H48 is a 32-year-old woman in the files against a
54-year-old man in the table (32 F is C48's entry). The files count 26 female and 33 male subjects, the
table 25 and 34, both within tolerance. The diagnoses differ in wording for many subjects (printed
above), as the files use a coarser coding.

## task-structure

> Each session has 100 encoding trials showing 100 novel images and 100 recognition trials, of which 50
> show new and 50 old images, and the images come from 5 visual categories.

Criterion: per session, 100 encoding trials with 100 distinct image files (external_image_file), 100
recognition trials with 50 of each label, and 5 categories; partially reproduced when each part holds in
more than half of the sessions.

In [ ]:
ts = []
for s in sessions:
    tr = s["trials"]
    learn = tr[tr.stim_phase == "learn"]
    r = C.recog(tr)
    ts.append(dict(identifier=s["identifier"], n_learn=len(learn), learn_files=learn.external_image_file.nunique(),
                   n_recog=len(r), n_label0=int((r.label == 0).sum()), n_label1=int((r.label == 1).sum()),
                   n_cat=tr.stimCategory.nunique(), n_cat_names=tr.category_name.nunique()))
ts = pd.DataFrame(ts)
ts["enc_ok"] = (ts.n_learn == 100) & (ts.learn_files == 100)
ts["rec_ok"] = (ts.n_recog == 100) & (ts.n_label0 == 50) & (ts.n_label1 == 50)
ts["cat_ok"] = (ts.n_cat == 5) & (ts.n_cat_names == 5)
print("sessions:", len(ts), " encoding part holds:", int(ts.enc_ok.sum()), " recognition part:", int(ts.rec_ok.sum()),
      " 5 categories:", int(ts.cat_ok.sum()), " all three:", int((ts.enc_ok & ts.rec_ok & ts.cat_ok).sum()))
print(ts[~(ts.enc_ok & ts.rec_ok & ts.cat_ok)].to_string())
print("encoding trials whose image file repeats within the block, all sessions:", int((ts.n_learn - ts.learn_files).sum()))

**Result: partially reproduced.** 79 of 87 sessions hold 100 encoding trials with 100 distinct images,
100 recognition trials with 50 of each label, and 5 categories. In 7 sessions the trial table holds only
50 encoding trials (150 trials in all), and in CS53_123 97. Every session has 100 recognition trials with
50 of each label and 5 categories, so each part holds in more than half of the sessions.

## new-old-label-coding

> In the trials table, new_old_labels_recog is 0 for old stimuli (seen during encoding) and 1 for new
> stimuli.

Criterion, per session: (1) under the stated coding, label 0 trials show image files from the session's
encoding block and label 1 trials do not; (2) label 0 (old) trials receive "old" responses (4 to 6) more
often than label 1 trials. Reproduced when both hold in every session; not reproduced when, in most
sessions, both hold with the labels reversed.

In [ ]:
lc = []
for s in sessions:
    tr = s["trials"]
    enc = set(tr[tr.stim_phase == "learn"].external_image_file)
    r = C.recog(tr)
    in0 = r[r.label == 0].external_image_file.isin(enc).mean()
    in1 = r[r.label == 1].external_image_file.isin(enc).mean()
    old0 = (r[r.label == 0].conf >= 4).mean()
    old1 = (r[r.label == 1].conf >= 4).mean()
    lc.append(dict(identifier=s["identifier"], n_learn=int((tr.stim_phase == "learn").sum()), label0_in_encoding=in0,
                   label1_in_encoding=in1, label0_old_resp=old0, label1_old_resp=old1))
lc = pd.DataFrame(lc)
lc["stated_1"] = (lc.label0_in_encoding == 1) & (lc.label1_in_encoding == 0)
lc["stated_2"] = lc.label0_old_resp > lc.label1_old_resp
lc["reversed_1"] = (lc.label1_in_encoding > 0) & (lc.label0_in_encoding == 0)
lc["reversed_1_full"] = (lc.label1_in_encoding == 1) & (lc.label0_in_encoding == 0)
lc["reversed_2"] = lc.label1_old_resp > lc.label0_old_resp
print("sessions where both tests hold under the stated coding (0 = old):", int((lc.stated_1 & lc.stated_2).sum()), "of", len(lc))
print("sessions where both hold reversed (1 = old):", int((lc.reversed_1 & lc.reversed_2).sum()),
      "; with every label-1 image in the encoding block:", int((lc.reversed_1_full & lc.reversed_2).sum()))
print("label-1 images found in the encoding block: median %.2f, min %.2f; label-0 images: max %.2f"
      % (lc.label1_in_encoding.median(), lc.label1_in_encoding.min(), lc.label0_in_encoding.max()))
print("'old' responses, median over sessions: label 1 %.2f, label 0 %.2f" % (lc.label1_old_resp.median(), lc.label0_old_resp.median()))
print("\nsessions where not every label-1 image is in the encoding block, or the response test fails:")
print(lc[~(lc.reversed_1_full & lc.reversed_2)].round(2).to_string())

**Result: not reproduced.** The coding is reversed: label 1 marks old images. In no session do label 0
trials show encoding images; label 1 trials do in all 87 sessions, and receive more "old" responses in 86
(both tests hold reversed in 86 of 87; in T98_2005 the response test fails, 0.68 against 0.72). In the 7
sessions whose trial table holds only 50 encoding trials, about half of the label 1 images (0.52 to 0.58)
are in the stored encoding block. The paper's code also treats 1 as old.

## ttl-codes

> The events stream holds the TTL markers 55, 1, 2, 3, 20 or 21, 31 to 36, 6 and 66 with the meanings of
> Table 1, and the events and experiment_ids streams have one entry each, in the same order.

Criterion: in every file, events and experiment_ids have the same length and timestamps, the events hold
only the listed codes, and every trial shows 1, 2, 3, a response of its phase's kind and 6 in this order;
partially reproduced when this holds in most files. A trial's events are those from its stimulus onset (1) to the
next stimulus onset, with the experiment markers 55 and 66 set aside (the k-th onset is the k-th trial).

In [ ]:
listed = {55, 1, 2, 3, 20, 21, 31, 32, 33, 34, 35, 36, 6, 66}
tt = []
for s in sessions:
    ev, et = s["events"].astype(int), s["events_t"]
    same = len(ev) == len(s["expids"]) and np.allclose(et, s["expids_t"])
    extra = sorted(set(np.unique(ev)) - listed)
    # trial events: the stream without the experiment markers 55 and 66, cut at each stimulus onset (1)
    e = ev[~np.isin(ev, [55, 66])]
    starts = np.where(e == 1)[0]
    segs = [e[a:b].tolist() for a, b in zip(starts, list(starts[1:]) + [len(e)])]
    phase = s["trials"].stim_phase.values
    bad, examples = 0, []
    for k, seg in enumerate(segs):
        resp = {20, 21} if k < len(phase) and phase[k] == "learn" else {31, 32, 33, 34, 35, 36}
        if not (len(seg) == 5 and seg[:3] == [1, 2, 3] and seg[3] in resp and seg[4] == 6):
            bad += 1
            examples.append(seg)
    tt.append(dict(identifier=s["identifier"], n_events=len(ev), same_length_and_times=same, extra_codes=extra,
                   trials=len(phase), onsets=len(segs), events_before_first_onset=e[:starts[0]].tolist() if len(starts) else e.tolist(),
                   trials_bad_sequence=bad, n55=int((ev == 55).sum()), n66=int((ev == 66).sum()),
                   example=examples[0] if examples else None))
tt = pd.DataFrame(tt)
tt["ok"] = (tt.same_length_and_times & (tt.extra_codes.str.len() == 0) & (tt.trials_bad_sequence == 0)
            & (tt.onsets == tt.trials) & (tt.events_before_first_onset.str.len() == 0))
print("files where events and experiment_ids have the same length and timestamps:", int(tt.same_length_and_times.sum()), "of", len(tt))
print("files with only the listed codes:", int((tt.extra_codes.str.len() == 0).sum()))
print("files with one stimulus onset per trial:", int((tt.onsets == tt.trials).sum()))
print("files where every trial has the sequence 1, 2, 3, response of its phase, 6:", int(((tt.trials_bad_sequence == 0) & (tt.events_before_first_onset.str.len() == 0)).sum()))
print("files meeting the whole criterion:", int(tt.ok.sum()))
print("trials failing the sequence, all files:", int(tt.trials_bad_sequence.sum()), "of", int(tt.trials.sum()))
print(tt[~tt.ok].drop(columns=["ok"]).to_string())
print("\ncounts of the unlisted codes over all files:", dict(Counter(int(x) for s in sessions for x in s["events"] if int(x) not in listed)))
print("files with two 55 markers:", int((tt.n55 == 2).sum()), "; with none:", tt[tt.n55 == 0].identifier.tolist(), "; with one:", tt[tt.n55 == 1].identifier.tolist())
print("files with two 66 markers:", int((tt.n66 == 2).sum()), "; others:", tt[tt.n66 != 2][["identifier", "n66"]].to_dict("records"))

**Result: partially reproduced.** In all 87 files the two streams have the same length and timestamps, and 78
files meet the whole criterion. Nine files hold codes not in Table 1 (0, 4, 22, 23, 98 and 99), and 30 of
17047 trials, all in those files, break the sequence: in T88_2001, T88_2002 and T90_2003 a second 3 comes
before the 2, and CS53_123 starts in the middle of a trial. Seven files have no 55 marker, which the
criterion does not require.

## experiment-ids

> The learning block is labelled with one of 80, 83 or 88 and the recognition block with one of 81, 84 or 89.

The learning block's id is the experiment id at the TTL-1 events of the learning trials, the recognition
block's at those of the recognition trials.

In [ ]:
ei = []
for s in sessions:
    ev, ex = s["events"], s["expids"]
    on_idx = np.where(ev == 1)[0]
    phase = s["trials"].stim_phase.values
    n = min(len(on_idx), len(phase))
    learn_ids = sorted(set(ex[on_idx[:n]][phase[:n] == "learn"].astype(int)))
    recog_ids = sorted(set(ex[on_idx[:n]][phase[:n] == "recog"].astype(int)))
    ei.append(dict(identifier=s["identifier"], learn=learn_ids, recog=recog_ids, all_ids=sorted(set(ex.astype(int))),
                   data_collection=s["data_collection"],
                   variant=",".join(map(str, sorted(C.variant_of(s["trials"].external_image_file))))))
ei = pd.DataFrame(ei)
ei["ok"] = ei.apply(lambda r: len(r.learn) == 1 and len(r.recog) == 1 and r.learn[0] in (80, 83, 88) and r.recog[0] in (81, 84, 89), axis=1)
ei["pair_ok"] = ei.apply(lambda r: r.ok and (r.learn[0], r.recog[0]) in ((80, 81), (83, 84), (88, 89)), axis=1)
print("files whose two ids fall in the sets:", int(ei.ok.sum()), "of", len(ei), "; whose pair is one of Table 2's:", int(ei.pair_ok.sum()))
print(ei[~ei.ok].to_string())
print("\npairs used:", Counter((tuple(a), tuple(b)) for a, b in zip(ei.learn, ei.recog)))
print("pair against task variant:", pd.crosstab(ei.variant, ei.learn.astype(str)).to_dict())

**Result: partially reproduced.** In 83 of 87 files both ids fall in the stated sets and form one of
Table 2's pairs. CS33_77 uses 81 and 82, CS39_100 and H51_105 82 and 83, and H42_55 80 and 82. The ids
do not follow the task variant: 7 of the 20 variant 2 sessions and 18 of the 25 variant 3 sessions use 80
and 81.

## session-dates

> session_start_time keeps only the year and month of the experiment, with the day set to the first of
> the month for all sessions.

In [ ]:
sd = pd.DataFrame([dict(identifier=s["identifier"], start=s["session_start_time"]) for s in sessions])
sd["date"] = pd.to_datetime(sd.start.str[:10])
sd["first"] = sd.date.dt.day == 1
print("files whose session_start_time falls on the first of a month:", int(sd["first"].sum()), "of", len(sd))
print(sd[~sd["first"]].to_string())
years = sd.date.dt.year
print("years:", years.min(), "to", years.max(), "; by year:", years.value_counts().sort_index().to_dict())
print("files dated before 2006 (the earliest file name otherwise):", sd[years < 2006].identifier.tolist())
print("all session_start_time values carry the time 00:00:00:", sd.start.str[11:19].eq("00:00:00").all())

**Result: partially reproduced.** 86 of 87 files are dated on the first of a month; H28_48 is dated
2010-03-13. H44_68 is dated 2000-01-01, six years before any other file and before the Huntington
recordings that surround it in the session numbering (2006 to 2016), so its year is likely a default.

## units-table

> The units table holds spike_times, waveform_mean_encoding, waveform_mean_recognition (sampled at 100 kHz),
> origClusterID, SNR, IsolDist and electrodes, with one electrode per neuron.

The column the paper calls IsolDist is named IsolationDist in the files, with the description "IsolDist";
it is taken as that column, as the names differ only in spelling.

In [ ]:
need = ["spike_times", "waveform_mean_encoding", "waveform_mean_recognition", "origClusterID", "SNR", "electrodes"]
has_cols = [all(c in s["unit_columns"] for c in need) and ("IsolDist" in s["unit_columns"] or "IsolationDist" in s["unit_columns"]) for s in sessions]
print("files holding every column (IsolDist as IsolationDist):", sum(has_cols), "of", len(sessions),
      "; files with a column named exactly IsolDist:", sum("IsolDist" in s["unit_columns"] for s in sessions))
print("units linked to exactly one electrode:", int((U.n_electrodes == 1).sum()), "of", len(U))
print("waveform sampling rate stated:", U.waveform_sampling_rate.unique(), "Hz (expected 100000 +/- 10000)")
for c in ["SNR", "IsolationDist"]:
    v = U[c]
    print(f"{c}: NaN {int(v.isna().sum())}, infinite {int(np.isinf(v).sum())}, finite {int(np.isfinite(v).sum())} of {len(v)};"
          f" finite range {v[np.isfinite(v)].min():.3g} to {v[np.isfinite(v)].max():.3g}")
print("units with a non-finite SNR or IsolationDist:", int((~np.isfinite(U.SNR) | ~np.isfinite(U.IsolationDist)).sum()),
      "; in files:", int(U[~np.isfinite(U.SNR) | ~np.isfinite(U.IsolationDist)].identifier.nunique()))
print("IsolationDist above 1e6:", int((U.IsolationDist > 1e6).sum()))
print("waveforms finite in every unit:", bool(U.waveform_mean_encoding_finite.all() and U.waveform_mean_recognition_finite.all()))

**Result: partially reproduced.** Every file holds the columns (IsolDist under the name IsolationDist),
every unit has one electrode, and the stated waveform sampling rate is 98.4 kHz, within tolerance. The
quality metrics are not finite for every unit: IsolationDist is NaN for 493 of 1864 units and SNR for 75,
SNR is infinite for 11 units, and 2 isolation distances are above 1e6, which is not a plausible value.

## electrodes-table

> The electrodes table gives each electrode's brain area, MNI coordinates, impedance, the filter applied
> before spike sorting (300 to 3000 Hz) and the original hardware channel.

In [ ]:
def empty(col):
    v = E[col]
    if v.dtype == object:
        return v.isna() | v.astype(str).str.strip().isin(["", "nan", "NaN", "None"])
    return v.isna()

ec = pd.DataFrame({c: dict(present_in_files=sum(c in s["electrode_columns"] for s in sessions),
                           empty=int(empty(c).sum()), electrodes=len(E))
                   for c in ["location", "x", "y", "z", "imp", "filtering", "origChannel"]}).T
print(ec.to_string())
print("filtering values:", E.filtering.value_counts().to_dict())
xyz_missing = E[E.x.isna()]
print("files with missing coordinates:", xyz_missing.identifier.nunique(), "; electrodes", len(xyz_missing))

**Result: not reproduced.** The impedance column is empty (NaN) for all 887 electrodes. Location,
filtering (300-3000Hz) and origChannel are filled for every electrode, and the MNI coordinates are
missing for 99 electrodes.

## isi-criterion

> Clusters were accepted as units only when fewer than 3% of their interspike intervals were shorter than 3 ms.

In [ ]:
isi_sorted = np.array([C.isi_fraction(sp, sort=True) for s in sessions for sp in s["spikes"]])
isi_stored = np.array([C.isi_fraction(sp, sort=False) for s in sessions for sp in s["spikes"]])
unsorted = [(s["identifier"], i) for s in sessions for i, sp in enumerate(s["spikes"]) if np.any(np.diff(sp) < 0)]
print("units:", len(isi_sorted), " with fewer than 2 spikes:", int(np.isnan(isi_sorted).sum()))
print("sorted: below 3%%: %d (%.2f%%); at or above: %d" % ((isi_sorted < 0.03).sum(), 100 * np.nanmean(isi_sorted < 0.03), (isi_sorted >= 0.03).sum()))
print("ISI<3ms fraction, sorted: median %.4f, 95th percentile %.4f, max %.3f" % (np.nanmedian(isi_sorted), np.nanpercentile(isi_sorted, 95), np.nanmax(isi_sorted)))
for thr in (0.01, 0.02, 0.03, 0.05):
    print(f"  units below {thr:.0%}: {(isi_sorted < thr).sum()}")
print("variant, as stored: below 3%%: %d; at or above: %d" % ((isi_stored < 0.03).sum(), (isi_stored >= 0.03).sum()))
print("units whose stored spike times are not in time order:", len(unsorted), unsorted)
UU = U.assign(isi=isi_sorted)
print(UU[UU.isi >= 0.03][["identifier", "id", "n_spikes", "isi"]].to_string())

**Result: partially reproduced.** With spike times sorted, 1858 of 1864 units (99.7%) have fewer than 3% of
intervals below 3 ms; 6 units do not, two of them with about 50%. That is at least 95%, the partial threshold. The spike
times of 13 units are not stored in time order; as stored, 1860 units fall below 3%.

## fig5a-mtl-locations

> All neurons were recorded from the medial temporal lobe, in the amygdala and the hippocampus.

In [ ]:
unit_loc = pd.Series([s["electrodes"].location.values[r] for s in sessions for r in s["units"].electrode_row.values])
print(unit_loc.value_counts().to_string())
mtl = unit_loc.str.contains("Amygdala|Hippocampus", case=False)
print("units in amygdala or hippocampus:", int(mtl.sum()), "of", len(unit_loc))

**Result: reproduced.** The electrode of every one of the 1864 units is in the left or right amygdala or
hippocampus.

## fig5c-auc-all

> Subjects distinguished new from old images; the behavioural AUC averaged over all sessions is 0.74 ± 0.10.

AUC per session as in the paper's code: label 1 as old (claims file), cumulative ROC over the six
confidence levels, rates adjusted for 0 and 1, trapezoid AUC; mean and standard deviation over the 87
sessions. The confidence response is response_value minus 30; it is checked here against the TTL codes.

In [ ]:
resp_check = [np.array_equal(s["events"][(s["events"] >= 31) & (s["events"] <= 36)] - 30, C.recog(s["trials"]).conf.values) for s in sessions]
print("files where response_value - 30 equals the TTL 31-36 codes of the events stream, in order:", sum(resp_check), "of", len(sessions))
beh = []
for s in sessions:
    r = C.recog(s["trials"])
    H, F = C.cumulative_roc(r.conf, r.label == 1)
    Hs, Fs = C.cumulative_roc(r.conf, r.label == 0)
    a = C.auc(H, F)
    beh.append(dict(identifier=s["identifier"], label=s["label"], site=s["site"], auc=a, auc_stated_coding=C.auc(Hs, Fs),
                    slope=C.zroc_slope(H, F), H=H, F=F, conf=r.conf.values, old=(r.label == 1).values))
beh = pd.DataFrame(beh)
print("AUC over %d sessions: mean %.3f, SD %.3f (expected 0.74 +/- 0.10 and 0.10 +/- 0.01); range %.3f to %.3f"
      % (len(beh), beh.auc.mean(), beh.auc.std(ddof=1), beh.auc.min(), beh.auc.max()))
print("sessions with AUC below 0.5:", beh[beh.auc < 0.5].identifier.tolist())
print("with the stated coding (0 = old) the mean AUC would be %.3f" % beh.auc_stated_coding.mean())
once = beh[[keep[i].any() for i in beh.identifier]]
print("variant, the two sessions whose every unit repeats another file left out (%d sessions): mean %.3f, SD %.3f" % (len(once), once.auc.mean(), once.auc.std(ddof=1)))
fig, ax = plt.subplots(figsize=(4, 3))
ax.hist(beh.auc, bins=20)
ax.set_xlabel("AUC")
ax.set_ylabel("sessions")
plt.show()

**Result: reproduced.** The mean AUC over the 87 sessions is 0.753 with a standard deviation of 0.094,
both within tolerance. The confidence responses equal the TTL codes in every file.

## fig5c-auc-sites

> Performance was similar at the two study sites, with an average AUC of 0.74 ± 0.09 for CS sessions and
> 0.69 ± 0.13 for TWH sessions.

CS sessions are those of subjects labelled H or C, TWH those labelled T (Figure 5 legend).

In [ ]:
g = beh.groupby("site").auc.agg(["count", "mean", lambda x: x.std(ddof=1)])
g.columns = ["sessions", "mean", "sd"]
print(g.round(3).to_string())
print("expected CS 0.74 +/- 0.09 (SD 0.09 +/- 0.009), TWH 0.69 +/- 0.13 (SD 0.13 +/- 0.013)")
w = stats.ttest_ind(beh.auc[beh.site == "CS"], beh.auc[beh.site == "TWH"], equal_var=False)
print("Welch's t-test CS against TWH: t = %.2f, p = %.3f, difference of means %.3f" % (w.statistic, w.pvalue, g.loc["CS", "mean"] - g.loc["TWH", "mean"]))
print("TWH sessions by subject:", beh[beh.site == "TWH"][["identifier", "auc"]].round(3).to_dict("records"))

**Result: reproduced.** CS sessions average 0.760 (SD 0.085, 78 sessions) and TWH sessions 0.688 (SD 0.138,
9 sessions); all four values are within tolerance. The sites do not differ by Welch's t-test (p = 0.16).

## fig5b-roc-asymmetric

> The ROC curve is asymmetric (p < 0.05), as expected for declarative memory.

Criterion: the z-ROC slope per session (regression of z(H) on z(F) over the five interior points), mean
below 1 with a two-sided one-sample t-test against 1 at p < 0.05. Sessions whose five false alarm rates
are all equal have no slope and are left out and counted. Variants: Wilcoxon signed-rank test, and the
slope of the pooled ROC (the counts summed over sessions, as the code's countersAv).

In [ ]:
sl = beh.slope
left_out = beh[sl.isna()].identifier.tolist()
x = sl.dropna()
t = stats.ttest_1samp(x, 1.0)
wz = stats.wilcoxon(x - 1.0)
print("sessions with a slope: %d; left out (all five false alarm rates equal): %d %s" % (len(x), len(left_out), left_out))
print("z-ROC slope: mean %.3f, SD %.3f, median %.3f, range %.3f to %.3f; below 1 in %d of %d sessions"
      % (x.mean(), x.std(ddof=1), x.median(), x.min(), x.max(), (x < 1).sum(), len(x)))
print("one-sample t-test against 1: t = %.2f, p = %.2g" % (t.statistic, t.pvalue))
print("variant, Wilcoxon signed-rank against 1: p = %.2g" % wz.pvalue)
conf_all = np.concatenate(beh.conf.values)
old_all = np.concatenate(beh.old.values)
Hp, Fp = C.cumulative_roc(conf_all, old_all)
print("variant, pooled ROC over all trials: slope %.3f; H %s; F %s" % (C.zroc_slope(Hp, Fp), Hp.round(3), Fp.round(3)))
for site, gg in beh.groupby("site"):
    y = gg.slope.dropna()
    print(f"  {site}: {len(y)} sessions, mean slope {y.mean():.3f}, t-test against 1 p = {stats.ttest_1samp(y, 1.0).pvalue:.2g}")

**Result: reproduced.** Over the 82 sessions with a slope, the mean z-ROC slope is 0.782, below 1
(t-test against 1, p = 4e-9; Wilcoxon p = 2e-8). The pooled ROC has a slope of 0.816. Five sessions with
equal false alarm rates at every criterion were left out.

## fig5d-confidence-accuracy

> Accuracy is significantly higher in high than in low confidence trials (P < 0.001, paired t test) across
> all sessions.

Readings of the split: (a) the fixed split of the MATLAB code, high = 1 or 6 and low = 2 to 5; (b) the
dynamic split of the Python code (helper.dynamic_split). Sessions without responses in one group are left
out and counted. Variants: the unpaired Welch test the Python code runs, the code's session inclusion rule
(AUC at least 0.6 and both high and low responses used for new and old), and the MATLAB dynamic split.

In [ ]:
acc = []
for _, b in beh.iterrows():
    fh, fl, nh, nl = C.accuracy_fixed(b.conf, b.old)
    dh, dl, mode, dnh, dnl = C.accuracy_dynamic(b.conf, b.old, "python")
    mh, ml, mmode, _, _ = C.accuracy_dynamic(b.conf, b.old, "matlab")
    acc.append(dict(identifier=b.identifier, site=b.site, fixed_high=fh, fixed_low=fl, n_high=nh, n_low=nl,
                    dyn_high=dh, dyn_low=dl, mode=mode, mat_high=mh, mat_low=ml, included=C.code_inclusion(b.conf, b.auc)))
acc = pd.DataFrame(acc)
for name, h, l in [("(a) fixed split", "fixed_high", "fixed_low"), ("(b) Python dynamic split", "dyn_high", "dyn_low"),
                   ("variant, MATLAB dynamic split", "mat_high", "mat_low")]:
    d = acc[[h, l, "included"]].dropna()
    pt = stats.ttest_rel(d[h], d[l])
    wt = stats.ttest_ind(d[h], d[l], equal_var=False)
    di = d[d.included]
    pti = stats.ttest_rel(di[h], di[l])
    print(f"{name}: {len(d)} sessions ({len(acc) - len(d)} left out for an empty group); high {100 * d[h].mean():.1f}% vs low {100 * d[l].mean():.1f}%;"
          f" high > low in {(d[h] > d[l]).sum()}; paired t = {pt.statistic:.2f}, p = {pt.pvalue:.2g};"
          f" Welch p = {wt.pvalue:.2g}; code's inclusion ({len(di)} sessions) paired p = {pti.pvalue:.2g}")
print("sessions left out under (a):", acc[acc.fixed_high.isna() | acc.fixed_low.isna()].identifier.tolist())
print("split modes chosen by the Python code:", acc["mode"].value_counts().to_dict())
for site, gg in acc.groupby("site"):
    d = gg[["fixed_high", "fixed_low"]].dropna()
    print(f"  {site} (fixed): {len(d)} sessions, high {100 * d.fixed_high.mean():.1f}% vs low {100 * d.fixed_low.mean():.1f}%, paired p = {stats.ttest_rel(d.fixed_high, d.fixed_low).pvalue:.2g}")

**Result: reproduced.** Under the fixed split, accuracy is 80.1% in high and 60.9% in low confidence
trials over 84 sessions (paired t-test p = 3e-16); under the Python dynamic split 78.2% against 60.8% over
82 sessions (p = 1e-11). The unpaired tests, the code's inclusion rule and the MATLAB split all give
p below 0.001.

## Selection of VS and MS cells

Firing rates of every unit in the 100 recognition trials, in the Methods' window (200 to 1200 ms after
stimulus onset) and the code's (200 to 1700 ms). VS: one-way ANOVA across the 5 categories, p < 0.05.
MS: two-tailed bootstrap of equal means (bootstrapMeanEqualTest.m, 1000 resamples), old (label 1)
against new (label 0) trials, p < 0.05; all 50 and 50 trials (Methods) and the code's correct trials only
(old answered 4 to 6, new answered 1 to 3). The bootstrap uses its own generator, seeded once (seed 2020).
The nulls shuffle the category labels, or the new/old labels, within each session (seed 7).

In [ ]:
rng_ms = np.random.default_rng(2020)
rows = []
t0 = time.time()
for s in sessions:
    r = C.recog(s["trials"])
    on, cat = r.start_time.values, r.stimCategory.values
    old = (r.label == 1).values
    correct = (((r.conf <= 3) & ~old) | ((r.conf >= 4) & old)).values
    for i, sp in enumerate(s["spikes"]):
        row = dict(identifier=s["identifier"], label=s["label"], site=s["site"], unit=i, kept_once=keep[s["identifier"]][i])
        for wn, w in (("m", C.WINDOW_METHODS), ("c", C.WINDOW_CODE)):
            rt = C.window_rates(sp, on, w)
            row["vs_" + wn] = C.anova_p(rt, cat)
            row["ms_" + wn] = C.bootstrap_mean_equal(rt[old], rt[~old], 1000, rng_ms)
            row["msc_" + wn] = C.bootstrap_mean_equal(rt[old & correct], rt[~old & correct], 1000, rng_ms)
        row["n_old_correct"], row["n_new_correct"] = int((old & correct).sum()), int((~old & correct).sum())
        rows.append(row)
cells = pd.DataFrame(rows)
print(f"{len(cells)} units in {time.time() - t0:.0f} s")
print("units with the same rate in every trial (ANOVA undefined, counted as not VS): Methods' window",
      int(cells.vs_m.isna().sum()), ", code's window", int(cells.vs_c.isna().sum()))

rng_null = np.random.default_rng(7)
null = []
for s in sessions:
    r = C.recog(s["trials"])
    on = r.start_time.values
    cat = rng_null.permutation(r.stimCategory.values)
    old = rng_null.permutation((r.label == 1).values)
    for sp in s["spikes"]:
        rt = C.window_rates(sp, on, C.WINDOW_METHODS)
        null.append(dict(vs=C.anova_p(rt, cat), ms=C.bootstrap_mean_equal(rt[old], rt[~old], 1000, rng_null)))
null = pd.DataFrame(null)
print("nulls (labels shuffled within session, Methods' window): VS %.2f%%, MS %.2f%%" % (100 * (null.vs < 0.05).mean(), 100 * (null.ms < 0.05).mean()))

## vs-cells-proportion

> 343 of the 1863 cells (18.4%) are visually selective, by a one-way ANOVA (P < 0.05) of the firing rate in
> recognition trials, in a 1 s window starting 200 ms after stimulus onset, across the 5 visual categories.

In [ ]:
N = len(cells)
def show(mask, label, n=None):
    n = len(mask) if n is None else n
    print(f"  {label}: {int(mask.sum())} of {n} ({100 * mask.sum() / n:.2f}%)")
print("VS cells (expected 343 +/- 34.3 and 18.4% +/- 1.84):")
show(cells.vs_m < 0.05, "Methods' window, every unit")
show(cells.vs_c < 0.05, "variant, code's window")
k = cells[cells.kept_once]
show(k.vs_m < 0.05, "variant, duplicated trains counted once")
show(null.vs < 0.05, "null, categories shuffled within session")
print("by threshold (Methods' window):", {a: int((cells.vs_m < a).sum()) for a in (0.01, 0.05, 0.1)})
by_subj = cells.groupby("label").agg(units=("unit", "size"), vs=("vs_m", lambda p: int((p < 0.05).sum())))
print("by subject: VS fraction median %.2f, range %.2f to %.2f; subjects with no VS cell: %d of %d"
      % ((by_subj.vs / by_subj.units).median(), (by_subj.vs / by_subj.units).min(), (by_subj.vs / by_subj.units).max(), (by_subj.vs == 0).sum(), len(by_subj)))

**Result: reproduced.** In the Methods' window 332 of 1864 units (17.8%) are VS cells, within tolerance of
343 and 18.4%. The code's window gives 343 (18.4%), and the shuffled null 4.8%.

## ms-cells-proportion

> 146 of the 1863 cells (7.8%) are memory selective, by a two-tailed bootstrap comparison of mean firing
> rates (1,000 runs, P < 0.05) between the 50 novel and 50 familiar recognition trials in a 1 s window
> starting 200 ms after stimulus onset.

Criterion: every unit, the Methods' window and all 50 and 50 trials. The code's choices are variants.
The bootstrap is repeated with four other seeds to show its resampling spread.

In [ ]:
print("MS cells (expected 146 +/- 14.6 and 7.8% +/- 0.78):")
show(cells.ms_m < 0.05, "Methods' window, all 50 and 50 trials")
show(cells.ms_m <= 0.05, "  the same with p <= 0.05, as the code's selection compares")
show(cells.msc_m < 0.05, "variant, correct trials only, Methods' window")
show(cells.ms_c < 0.05, "variant, all trials, code's window")
show(cells.msc_c < 0.05, "variant, correct trials only, code's window (the code's choices)")
show(k.ms_m < 0.05, "variant, duplicated trains counted once")
show(null.ms < 0.05, "null, new/old labels shuffled within session")
print("correct trials per session: old %d to %d, new %d to %d"
      % (cells.n_old_correct.min(), cells.n_old_correct.max(), cells.n_new_correct.min(), cells.n_new_correct.max()))
def n_ms(seed, window, correct_only):
    rg = np.random.default_rng(seed)
    n = 0
    for s in sessions:
        r = C.recog(s["trials"])
        old = (r.label == 1).values
        use = (((r.conf <= 3) & ~old) | ((r.conf >= 4) & old)).values if correct_only else np.ones(len(r), bool)
        for sp in s["spikes"]:
            rt = C.window_rates(sp, r.start_time.values, window)
            n += C.bootstrap_mean_equal(rt[old & use], rt[~old & use], 1000, rg) < 0.05
    return int(n)
for label, w, co in (("main reading", C.WINDOW_METHODS, False), ("correct trials, Methods' window", C.WINDOW_METHODS, True),
                     ("the code's choices", C.WINDOW_CODE, True)):
    print(f"{label}, seeds 1 to 4:", [n_ms(seed, w, co) for seed in (1, 2, 3, 4)])
for site, gg in cells.groupby("site"):
    print(f"  {site}: {int((gg.ms_m < 0.05).sum())} of {len(gg)} ({100 * (gg.ms_m < 0.05).mean():.1f}%)")

**Result: not reproduced.** With all 50 and 50 trials in the Methods' window, 110 of 1864 units (5.9%) are
MS cells, below the tolerance of 146 (131 to 161); four other seeds give 109 to 111. With the paper's code's
choices (correct trials only, 200 to 1700 ms) 147 units (7.9%) qualify, 143 to 147 over the other seeds, a
match. Correct trials in the Methods' window give 132 to 147 depending on the seed. The shuffled null gives
4.4%.

## dual-cells-proportion

> 26 cells (1.4%) are both visually and memory selective.

In [ ]:
print("dual cells (expected 26 +/- 2.6 and 1.4% +/- 0.14):")
show((cells.vs_m < 0.05) & (cells.ms_m < 0.05), "both main tests")
show((cells.vs_c < 0.05) & (cells.msc_c < 0.05), "variant, the code's choices for both")
show((cells.vs_m < 0.05) & (cells.msc_m < 0.05), "variant, Methods' window, MS on correct trials")
show((null.vs < 0.05) & (null.ms < 0.05), "null")
print("expected by independence from the main counts: %.1f" % ((cells.vs_m < 0.05).mean() * (cells.ms_m < 0.05).mean() * N))

**Result: not reproduced.** 12 units (0.64%) are selected by both main tests, against 26 (1.4%). With the
code's choices for both tests 26 units (1.39%) qualify, as the paper states.

## fig5f-site-proportions

> The percentages of VS, MS and dual cells are comparable between CS sessions (18.4%, 7.7% and 1.4%) and the
> new TWH sessions (19.4%, 10.4% and 1.3%).

Criterion: two-sided Fisher's exact test between sites for each type at 0.05; reproduced when no test finds
a difference and at least four of the six values are within tolerance; partially reproduced when only one
of these holds.

In [ ]:
expected = {("CS", "VS"): 18.4, ("CS", "MS"): 7.7, ("CS", "dual"): 1.4, ("TWH", "VS"): 19.4, ("TWH", "MS"): 10.4, ("TWH", "dual"): 1.3}
def site_table(vs, ms, label):
    sel = {"VS": vs, "MS": ms, "dual": vs & ms}
    out, fisher = [], {}
    for typ, m in sel.items():
        tab = []
        for site in ("CS", "TWH"):
            mm = m[cells.site == site]
            pct = 100 * mm.mean()
            e = expected[(site, typ)]
            out.append(dict(site=site, type=typ, n=int(mm.sum()), units=len(mm), pct=round(pct, 2), expected=e,
                            within=abs(pct - e) <= 0.1 * e))
            tab.append([int(mm.sum()), int(len(mm) - mm.sum())])
        fisher[typ] = stats.fisher_exact(tab).pvalue
    out = pd.DataFrame(out)
    print(label)
    print(out.to_string(index=False))
    print("  Fisher's exact p:", {k: round(v, 3) for k, v in fisher.items()}, "; values within tolerance:", int(out.within.sum()), "of 6")
    return out, fisher
main_site, main_fisher = site_table(cells.vs_m < 0.05, cells.ms_m < 0.05, "main (Methods' window, all trials):")
site_table(cells.vs_c < 0.05, cells.msc_c < 0.05, "variant, the code's choices:")
site_table(cells.vs_m < 0.05, cells.msc_m < 0.05, "variant, Methods' window, MS on correct trials:")
print("TWH units by session:", sess[sess.site == "TWH"].set_index("identifier").n_units.to_dict())

**Result: partially reproduced.** No Fisher's exact test finds a difference between the sites (p = 0.36,
0.46 and 1.0 in the main reading), but only 1 of the 6 percentages is within tolerance: CS has 17.6% VS,
5.8% MS and 0.67% dual cells, TWH 22.1%, 7.8% and 0%. TWH has 77 units, so one cell moves its percentages by
1.3 points. With the code's choices all 6 values are within tolerance (CS 18.35%, 7.78%, 1.40%; TWH 19.48%,
10.39%, 1.30%) and no test finds a difference.

## twh-sessions

> The dataset includes 9 sessions from a new institution, Toronto Western Hospital.

In [ ]:
twh = sess[sess.label.str.startswith("T")]
print("sessions with a subject label starting with T:", len(twh), "(expected 9 +/- 0.9)")
print(twh[["identifier", "subject_id", "start", "n_units"]].to_string(index=False))
print("institution field of these files:", sorted({s["institution"] for s in sessions if s["label"].startswith("T")}))

**Result: reproduced.** Nine sessions, from eight subjects, have subject labels starting with T, and their
files name Toronto Western Hospital as the institution.

## previous-release-link

> The sessions of the earlier release (Faraut et al. 2018) are found in this dandiset under the same subject,
> session id and task variant.

Criterion: sessions matched by session id; a matched session agrees when its subject label and task variant
(from the image folder of external_image_file) are those of Faraut et al.'s Table 1. Reproduced when every
session is found and agrees; partially reproduced when at least 90% are.

In [ ]:
dandi_sess = pd.DataFrame([dict(session=s["sid"], label_d=s["label"], variant_d=sorted(C.variant_of(s["trials"].external_image_file)),
                                age_d=int(s["subject"]["age"]), sex_d=s["subject"]["sex"], units=len(s["units"])) for s in sessions])
lk = F1.merge(dandi_sess, on="session", how="left")
lk["found"] = lk.label_d.notna()
lk["agrees"] = lk.found & (lk.label == lk.label_d) & lk.apply(lambda r: r.found and r.variant_d == [r.variant], axis=1)
print("sessions in Faraut et al. Table 1:", len(lk), "; found:", int(lk.found.sum()), "; found and agree:", int(lk.agrees.sum()),
      "(%.1f%%)" % (100 * lk.agrees.mean()))
print("not found or not agreeing:")
print(lk[~lk.agrees][["label", "session", "variant", "age", "sex", "label_d", "variant_d"]].to_string(index=False))
fa = lk[lk.found]
print("matched sessions whose age differs:", fa[fa.age != fa.age_d][["label", "session", "age", "age_d"]].to_dict("records"))
print("matched sessions whose sex differs:", fa[fa.sex != fa.sex_d][["label", "session", "sex", "sex_d"]].to_dict("records"))
print("Faraut's table against this paper's Online-only Table 1, for H48:", F1[F1.label == "H48"][["session", "variant", "age", "sex"]].to_dict("records"),
      T1[T1.label == "H48"][["session", "variant", "age", "sex"]].to_dict("records"))

**Result: partially reproduced.** 64 of the 65 sessions of the earlier release are found under the same
subject and variant (98.5%). Session 98 of subject H48 is not in the dandiset, nor in this paper's
Online-only Table 1. Among matched sessions, ages differ for C24 (47 against 57) and H33, and H48's
session 99 is a 32-year-old woman in the file against a 54-year-old man in both papers.

## previous-release-additions

> Relative to the earlier release of this task's data (Faraut et al. 2018), this release adds 23 sessions,
> 17 subjects and 288 neurons.

Criterion: new sessions are the dandiset's sessions not in Faraut et al.'s Table 1, new subjects those with
no session in it, new neurons the unit rows of the new sessions. Reproduced when all three are within 10%;
partially reproduced when one or two are.

In [ ]:
new_s = dandi_sess[~dandi_sess.session.isin(F1.session)]
old_labels = set(dandi_sess[dandi_sess.session.isin(F1.session)].label_d)
new_subj = sorted(set(new_s.label_d) - old_labels)
print("new sessions:", len(new_s), "(expected 23 +/- 2.3):", sorted(new_s.session))
print("new subjects:", len(new_subj), "(expected 17 +/- 1.7):", new_subj)
print("new neurons (unit rows of new sessions):", int(new_s.units.sum()), "(expected 288 +/- 28.8)")
print("  by site:", new_s.assign(site=np.where(new_s.label_d.str.startswith("T"), "TWH", "CS")).groupby("site").units.sum().to_dict())
n_old_rel = dandi_sess[dandi_sess.session.isin(F1.session)].units.sum()
print("unit rows of the sessions shared with the earlier release:", int(n_old_rel), "(Faraut et al. abstract: 1576 neurons)")
print("variant, differences of the stated totals: sessions 87 - 65 =", 87 - 65, "; subjects 59 - 42 =", 59 - 42, "; neurons 1863 - 1576 =", 1863 - 1576)
print("variant, differences of the counts here: sessions", len(dandi_sess) - len(F1), "; subjects", dandi_sess.label_d.nunique() - F1.label.nunique(),
      "; neurons", int(dandi_sess.units.sum()) - 1576)

**Result: partially reproduced.** The dandiset adds 23 sessions and 17 subjects to the earlier release, as
stated, but those sessions hold 325 unit rows, above the tolerance of 288 (259 to 317). The sessions shared
with the earlier release hold 1539 unit rows, 37 fewer than its stated 1576 neurons.

## fig4-example-vs-neuron

> Neuron 132-177-4-RA (session 132, channel 177, cell 4, right amygdala) is visually selective, responding
> most to images of animals.

Criterion: the unit on channel 177 of session 132 with electrode row 4, in the right amygdala; ANOVA across
the 5 categories of recognition trials p < 0.05 (Methods' window) with animals the highest mean rate. The
code's window is a variant.

In [ ]:
s132 = [s for s in sessions if s["sid"] == 132][0]
e, u = s132["electrodes"], s132["units"]
u = u.assign(channel=e.origChannel.values[u.electrode_row.values], location=e.location.values[u.electrode_row.values])
print(s132["identifier"], "units:")
print(u[["id", "origClusterID", "electrode_row", "channel", "location", "n_spikes"]].to_string(index=False))
cand = u[(u.channel == 177) & (u.electrode_row == 4)]
print("units on channel 177 with electrode row 4:", len(cand))
i = int(cand.index[0])
r = C.recog(s132["trials"])
names = dict(zip(r.stimCategory, r.category_name))
for wl, w in (("Methods' window", C.WINDOW_METHODS), ("code's window", C.WINDOW_CODE)):
    rt = C.window_rates(s132["spikes"][i], r.start_time.values, w)
    p = C.anova_p(rt, r.stimCategory.values)
    means = pd.Series(rt).groupby(r.stimCategory.values).mean().rename(index=names)
    print(f"{wl}: ANOVA p = {p:.2g}; mean rate by category (Hz): {means.round(2).to_dict()}; highest: {means.idxmax()}")

**Result: reproduced.** Session 132 (CS58_132) holds one unit on channel 177, at electrode row 4 in the right
amygdala. Its recognition rate differs across categories (ANOVA p = 1.2e-22 in the Methods' window, 7.9e-19
in the code's) and is highest for animals (6.75 Hz, against 0.85 Hz or less for the other categories).

## fig3-example-vs-neuron

> Neuron 144-140-3-RH (session 144, channel 140, cell 3, right hippocampus) is visually selective for phones.

In [ ]:
print("session 144 among the dandiset's sessions:", 144 in set(sess.session), "; in Online-only Table 1:", 144 in set(T1.session),
      "; in Faraut et al. Table 1:", 144 in set(F1.session))
print("highest session id below 2000 in the dandiset:", sess[sess.session < 2000].session.max())

**Result: blocked (data missing).** Session 144 is not in the dandiset, nor in either paper's session table,
so the neuron cannot be found.

## Summary

In [ ]:
print("notebook ran in %.1f min; peak memory %.2f GB" % ((time.time() - started) / 60, resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6))

| Claim | Outcome |
|---|---|
| counts-neurons-subjects | reproduced (1864 unit rows, 59 subjects) |
| counts-sessions | reproduced (the 87 sessions of Online-only Table 1) |
| table1-demographics | partially reproduced (3 of 59 subjects differ in age or sex) |
| task-structure | partially reproduced (8 sessions lack encoding trials) |
| new-old-label-coding | not reproduced (label 1 is old) |
| ttl-codes | partially reproduced (9 files hold unlisted codes) |
| experiment-ids | partially reproduced (4 files use other ids) |
| session-dates | partially reproduced (one file on the 13th; one dated 2000) |
| units-table | partially reproduced (quality metrics missing or not finite) |
| electrodes-table | not reproduced (impedance empty for every electrode) |
| isi-criterion | partially reproduced (6 of 1864 units at or above 3%) |
| fig5a-mtl-locations | reproduced |
| fig5c-auc-all | reproduced (0.753 ± 0.094) |
| fig5c-auc-sites | reproduced (CS 0.760 ± 0.085, TWH 0.688 ± 0.138) |
| fig5b-roc-asymmetric | reproduced (mean z-ROC slope 0.78) |
| fig5d-confidence-accuracy | reproduced (paired p below 0.001 under both splits) |
| vs-cells-proportion | reproduced (332, 17.8%) |
| ms-cells-proportion | not reproduced (110, 5.9%; the code's choices give 147) |
| dual-cells-proportion | not reproduced (12; the code's choices give 26) |
| fig5f-site-proportions | partially reproduced (no difference between sites; 1 of 6 values within tolerance) |
| twh-sessions | reproduced (9) |
| previous-release-link | partially reproduced (64 of 65 sessions) |
| previous-release-additions | partially reproduced (23 sessions and 17 subjects; 325 neurons) |
| fig4-example-vs-neuron | reproduced |
| fig3-example-vs-neuron | blocked: session 144 is not in the dandiset |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the
differences from the paper's method.

How the analysis differs from the paper's: the AUC, z-ROC and accuracy computations follow the paper's code,
with label 1 taken as old as the code does. The VS and MS tests follow the Methods (a 1 s window, and all 50
and 50 trials for MS) as the claims file asks; the paper's code uses a 1.5 s window and correct trials only
for MS, and with those choices the counts and the per-site percentages of Figure 5f are matched.
Every unit row is counted, as the paper counts neurons per session; the units of the duplicated sessions
are shown counted once as a variant.

What was found about the dataset: the new/old label coding is the reverse of the paper's text and the
column's description; the impedance column is empty; IsolationDist and SNR are missing or not finite for
many units; 8 sessions hold fewer than 100 encoding trials; some event streams hold codes outside Table 1
and four files use experiment ids outside Table 2; two session dates break the stated rule; the spike times
of 13 units are not in time order; sessions 43 and 44 and sessions 54 and 55 hold the same spike trains for
every unit, and sessions 38 and 39 share 11 of their units, each pair from one continuous recording of two
sessions; the subject metadata of H48 (32, F) disagrees
with both papers (54, M); and session 98 of the earlier release is not in the dandiset.